# UTS Temu Kembali Informasi 2026 — SUBCPMK-4: Preprocessing, Extracting, dan Indexing Putusan Narkotika

**Mata kuliah:** Temu Kembali Informasi · Semester Ganjil 2026/2027 · Kelas A,D

| Anggota | NIM |
| --- | --- |
| Rizqullah Zuhayr Alfito Khan | 202310370311124 |
| Fakhri Ziddan Akbar | 202310370311143 |

**Dataset sendiri:** 50 putusan PN Kuala Kurun · Pidana Khusus · Narkotika dan Psikotropika · 2023–2025, dibaca langsung dari repository GitHub kelompok ([Dataset-Narkotika_124_143](https://github.com/FakhriZiddanAkbar/Dataset-Narkotika_124_143)).

**Alur notebook**
1. **Akses dataset** — mengunduh `Narkotika.zip` dari GitHub dan mengekstrak 50 PDF.
2. **Extracting** — teks *seluruh* putusan diambil dari PDF (lapisan watermark Mahkamah Agung dibuang), lalu dibagi menjadi empat bidang: kepala (identitas dan tuntutan), pertimbangan (dakwaan, saksi, fakta, pertimbangan hukum), amar, dan penutup. Metadata (terdakwa, pekerjaan, pidana, denda) diekstrak dengan regex.
3. **Preprocessing** — casefolding, normalisasi angka dan varian istilah, tokenisasi, *stopword removal*, dan *stemming* bahasa Indonesia. Tiap tahap dapat dihidupkan/dimatikan agar dampaknya terukur.
4. **Indexing** — *inverted index* posisional yang ditulis sendiri (tanpa Whoosh/Elasticsearch/rank_bm25).
5. **Pembobotan dan model pencarian** — TF mentah, TF-IDF + cosine (model ruang vektor), BM25 (model probabilistik), BM25 berbobot bidang, serta model Boolean dan pencarian frasa.
6. **Evaluasi** — 12 kueri dengan penilaian relevansi otomatis; membandingkan tiap tahap praproses dan tiap skema pembobotan (MAP, nDCG@10, R-Precision, MRR).

**Catatan keaslian:** seluruh kode ditulis kelompok sendiri. `scikit-learn` hanya dipakai untuk *memvalidasi* bahwa TF-IDF buatan sendiri menghasilkan skor yang sama, bukan untuk membangun mesin pencari.

## 1. Instalasi, konfigurasi, dan akses dataset
Semua pengaturan ada di sel konfigurasi. Dataset diunduh dari GitHub (repository harus **Public**).

In [ ]:
!pip -q install pdfplumber PySastrawi

In [ ]:
# ===== IMPOR DAN KONFIGURASI (ubah di sini saja) =====
import os, re, io, json, math, time, glob, zipfile, shutil, pickle, warnings
from collections import Counter, defaultdict

import requests
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib
import matplotlib.pyplot as plt
import pdfplumber
from tqdm.auto import tqdm
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory

try:
    from IPython.display import display
except Exception:          # dijalankan di luar Jupyter
    display = print

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 150)
pd.set_option("display.width", 200)

# --- Sumber dataset (repository GitHub kelompok; SUBCPMK-2) ---
DATASET_URLS = [
    "https://raw.githubusercontent.com/FakhriZiddanAkbar/Dataset-Narkotika_124_143/main/Dataset/Narkotika.zip",
    "https://github.com/FakhriZiddanAkbar/Dataset-Narkotika_124_143/raw/main/Dataset/Narkotika.zip",
]
OVERVIEW_URL = "https://raw.githubusercontent.com/FakhriZiddanAkbar/Dataset-Narkotika_124_143/main/Overview/Overview.xlsx"
DATASET_ZIP_LOKAL = ""            # opsional: path Narkotika.zip di Colab/Drive bila GitHub tidak dapat diakses
JUMLAH_DOKUMEN_DIHARAPKAN = 50

# --- Parameter model ---
BM25_K1, BM25_B = 1.5, 0.75       # parameter BM25 (nilai baku literatur)
BOBOT_BIDANG = {"kepala": 2.0, "pertimbangan": 1.0, "amar": 2.0, "penutup": 0.5}   # untuk BM25 berbobot bidang
TOP_K = 5                         # jumlah hasil pencarian yang ditampilkan

# --- Keluaran ---
DI_COLAB = os.path.isdir("/content")
OUT_DIR = "/content/hasil_subcpmk4" if DI_COLAB else os.path.abspath("hasil_subcpmk4")
DATA_DIR = os.path.join(OUT_DIR, "dataset")
TXT_DIR = os.path.join(OUT_DIR, "txt")
SIMPAN_KE_DRIVE = False           # True -> salin hasil ke Google Drive
DRIVE_FOLDER = "UTS_IR_SUBCPMK4"
for _d in (OUT_DIR, TXT_DIR):
    os.makedirs(_d, exist_ok=True)
print("Folder keluaran:", OUT_DIR)

In [ ]:
# ===== AKSES DATASET: Narkotika.zip dari GitHub -> 50 PDF putusan =====
def _zip_valid(isi):
    try:
        with zipfile.ZipFile(io.BytesIO(isi)) as z:
            return z.testzip() is None and any(n.lower().endswith(".pdf") for n in z.namelist())
    except zipfile.BadZipFile:
        return False


def ambil_dataset():
    zip_bytes, sumber = None, None
    if DATASET_ZIP_LOKAL and os.path.isfile(DATASET_ZIP_LOKAL):
        with open(DATASET_ZIP_LOKAL, "rb") as f:
            zip_bytes, sumber = f.read(), DATASET_ZIP_LOKAL
    else:
        for url in DATASET_URLS:
            for percobaan in range(3):
                try:
                    r = requests.get(url, timeout=120)
                    if r.status_code == 200 and _zip_valid(r.content):
                        zip_bytes, sumber = r.content, url
                        break
                except requests.RequestException:
                    pass
                time.sleep(3)
            if zip_bytes:
                break
    if zip_bytes is None:
        raise RuntimeError(
            "Dataset tidak dapat diunduh dari GitHub. Pastikan repository berstatus Public, atau unggah "
            "Narkotika.zip ke Colab (ikon folder di kiri) lalu isi DATASET_ZIP_LOKAL = '/content/Narkotika.zip'.")
    shutil.rmtree(DATA_DIR, ignore_errors=True)
    os.makedirs(DATA_DIR)
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
        z.extractall(DATA_DIR)
    pdfs = sorted(glob.glob(os.path.join(DATA_DIR, "**", "*.pdf"), recursive=True))
    print(f"Sumber dataset : {sumber}")
    print(f"Ukuran ZIP     : {len(zip_bytes)/1e6:.1f} MB | jumlah PDF: {len(pdfs)}")
    if len(pdfs) != JUMLAH_DOKUMEN_DIHARAPKAN:
        print(f"PERHATIAN: jumlah PDF {len(pdfs)} berbeda dari yang diharapkan ({JUMLAH_DOKUMEN_DIHARAPKAN}).")
    return pdfs


PDF_FILES = ambil_dataset()

# Metadata pendamping (Overview.xlsx) -> dipakai hanya untuk pemeriksaan silang nomor putusan
try:
    df_overview = pd.read_excel(OVERVIEW_URL)
    print("Overview.xlsx  :", df_overview.shape, "| kolom:", list(df_overview.columns))
except Exception as e:
    df_overview = None
    print("Overview.xlsx tidak dapat dibaca (dilewati):", str(e)[:80])

## 2. Extracting: PDF → teks penuh → bidang dan metadata
Teks diekstrak dengan `pdfplumber`. Karakter lapisan *watermark* dan *disclaimer* Mahkamah Agung (font Helvetica) disaring sebelum teks dibentuk, sehingga kata tidak terpecah oleh huruf lepas. Tiap putusan lalu dibagi menjadi **empat bidang yang saling lepas dan menutup seluruh teks**, sehingga tidak ada bagian putusan yang terbuang.

In [ ]:
# ===== PDF -> TEKS -> FIELD (model ekstraksi berbasis regex buatan sendiri) =====

def teks_pdf(path):
    """Ekstrak teks PDF. Karakter lapisan watermark/catatan kaki Mahkamah Agung
    (font Helvetica) dibuang agar kata tidak terpecah oleh huruf lepas."""
    halaman = []
    with pdfplumber.open(path) as pdf:
        for p in pdf.pages:
            q = p.filter(lambda o: o.get("object_type") != "char"
                         or not o["fontname"].split("+")[-1].startswith("Helvetica"))
            halaman.append(q.extract_text() or "")
    return "\n".join(halaman)


_RE_HDR_BARIS = re.compile(
    r"^\s*(Halaman\s+\d+(\s+dari\s+\d+)?(\s+Putusan\s+Nomor\s+\S+)?"
    r"|Putusan\s+Nomor\s+\S+(\s+Halaman.*)?)\s*$", re.I)


def bersihkan(t):
    """Rapikan teks: hapus penanda halaman, sisa disclaimer/watermark, dan spasi ganda."""
    baris = [b for b in t.split("\n") if not _RE_HDR_BARIS.match(b)]
    t = " ".join(baris)
    t = re.sub(r"\s*Halaman\s+\d+\s+dari\s+\d+\s+Putusan\s+Nomor\s+\d+/Pid\.Sus/\d{4}/PN\s*\w+\s*", " ", t)
    t = re.sub(r"Disclaimer\s+Kepaniteraan.*?\(ext\.?\s*318\)", " ", t, flags=re.S)
    t = re.sub(r"Direktori\s+Putusan\s+Mahkamah\s+Agung\s+Republik\s+Indonesia", " ", t)
    t = re.sub(r"putusan\.mahkamahagung\.go\.id", " ", t)
    t = re.sub(r"\s+", " ", t).strip()
    t = re.sub(r"(?<=\s)-(?=[A-Za-z0-9(])", " - ", t)   # butir "-1 (satu)" -> "- 1 (satu)"
    return re.sub(r"\s+", " ", t).strip()


_RE_MENGADILI = re.compile(r"M\s?E\s?N\s?G\s?A\s?D\s?I\s?L\s?I\s*:?")
_RE_DEMIKIAN = re.compile(r"Demikian(lah)?\s+(putusan\s+)?(ini\s+)?(di|telah)")


def ekstrak_amar(teks):
    """Amar = dari butir pertama setelah MENGADILI (kemunculan terakhir) sampai 'Demikianlah diputus'."""
    posisi = [m.end() for m in _RE_MENGADILI.finditer(teks)]
    if not posisi:
        return ""
    s = teks[posisi[-1]:]
    e = _RE_DEMIKIAN.search(s)
    if e:
        s = s[:e.start()]
    return bersihkan(s)


def ekstrak_barang_bukti(amar):
    """Barang bukti = daftar setelah 'Menetapkan barang bukti berupa' sampai butir amar berikutnya."""
    m = re.search(r"barang\s+bukti\s+berupa\s*[:;]?", amar, re.I)
    if not m:
        return ""
    sisa = amar[m.end():]
    e = re.search(r"\s\d{1,2}\.\s*(Membebankan|Menetapkan|Memerintahkan|Menghukum)", sisa)
    return (sisa[:e.start()] if e else sisa).strip()


def ekstrak_identitas(teks):
    """Identitas terdakwa dari blok 'Nama lengkap : ... Pekerjaan : ...' di awal putusan."""
    awal = teks[:6000]
    def ambil(pola):
        m = re.search(pola, awal, re.I | re.S)
        return re.sub(r"\s+", " ", m.group(1)).strip(" ;.") if m else ""
    return {
        "Terdakwa": ambil(r"Nama\s+lengkap\s*:\s*(.+?)(?=\s*\n\s*(?:\d\.\s*)?Tempat\s+lahir)"),
        "Umur": ambil(r"Umur[^:\n]*:\s*([^\n;]+)"),
        "Jenis Kelamin": ambil(r"Jenis\s+kelamin\s*:\s*([^\n;]+)"),
        "Pekerjaan": ambil(r"Pekerjaan\s*:\s*([^\n;]+)"),
    }


def ekstrak_tuntutan(teks):
    """Ringkasan tuntutan pidana Penuntut Umum (blok 'tuntutan pidana ... sebagai berikut:')."""
    m = re.search(r"tuntutan\s+pidana[^:]{0,120}?(?:sebagai\s+berikut|berikut)\s*:", teks, re.I | re.S)
    if not m:
        return ""
    blok = teks[m.end(): m.end() + 3000]
    e = re.search(r"\n\s*(?:Menimbang|Setelah\s+mendengar|Atas\s+tuntutan|Terdakwa\s+(?:dan|melalui|telah)|Penasihat)", blok)
    if e:
        blok = blok[:e.start()]
    return bersihkan(blok)[:1500]


def ekstrak_pidana(amar):
    pen = re.search(r"pidana\s+penjara\s+selama\s+(.+?)(?=\s+dan\s+(?:pidana\s+)?denda|\s+serta|;|\s+Menetapkan|\s+\d{1,2}\.\s|$)", amar, re.I)
    den = re.search(r"denda\s+(?:sebesar|sejumlah)\s+(Rp\s?[\d\.\,]+)", amar, re.I)
    return (pen.group(1).strip(" ,.") if pen else ""), (den.group(1).replace(" ", "").rstrip(".,") if den else "")


def ekstrak_tanggal_putusan(teks):
    t = bersihkan(teks)
    m = _RE_DEMIKIAN.search(t)
    if not m:
        return ""
    k = re.search(r"tanggal\s+(\d{1,2}\s+[A-Za-z]+\s+\d{4})", t[m.start(): m.start() + 600])
    return k.group(1) if k else ""


def ekstrak_nomor_dan_lembaga(teks):
    awal = teks[:3000]
    n = re.search(r"Nomor\s+(\d+\s*/\s*Pid\.\s*Sus\s*/\s*\d{4}\s*/\s*PN\s*\w+)", awal, re.I)
    nomor = re.sub(r"\s*/\s*", "/", n.group(1)).replace("Pid. Sus", "Pid.Sus") if n else ""
    l = re.search(r"Pengadilan\s+Negeri\s+([A-Z][A-Za-z ]+?)(?=\s+(?:yang|Klas|Kelas)|\s*[,\n.])", awal)
    lembaga = ("PN " + l.group(1).strip()) if l else ""
    return nomor, lembaga


def ekstrak_dari_teks(teks):
    """Satu putusan -> satu baris (dict). Lima kolom pertama = format wajib soal."""
    nomor, lembaga = ekstrak_nomor_dan_lembaga(teks)
    amar = ekstrak_amar(teks)
    penjara, denda = ekstrak_pidana(amar)
    baris = {
        "No Putusan": nomor,
        "Lembaga Peradilan": lembaga,
        "Barang Bukti": ekstrak_barang_bukti(amar),
        "Amar Putusan": amar,
    }
    baris.update(ekstrak_identitas(teks))
    baris.update({
        "Tuntutan": ekstrak_tuntutan(teks),
        "Pidana Penjara": penjara,
        "Denda": denda,
        "Tanggal Putusan": ekstrak_tanggal_putusan(teks),
        "Tahun": (re.search(r"/(\d{4})/", nomor).group(1) if re.search(r"/(\d{4})/", nomor) else ""),
    })
    return baris

In [ ]:
# ===== MEMBANGUN KORPUS: teks penuh + bidang + metadata untuk setiap putusan =====
def segmentasi(t):
    """Bagi SELURUH teks putusan (sudah bersih) menjadi 4 bidang yang saling lepas dan utuh:
       kepala       : identitas terdakwa, tuntutan, hingga sebelum 'Menimbang' pertama
       pertimbangan : dakwaan, keterangan saksi/ahli, fakta, dan pertimbangan hukum
       amar         : butir-butir setelah 'MENGADILI' sampai sebelum 'Demikianlah'
       penutup      : susunan majelis, tanggal putusan, panitera"""
    m1 = re.search(r"\bMenimbang\b", t)
    i_pert = m1.start() if m1 else min(6000, len(t))
    ms = list(_RE_MENGADILI.finditer(t))
    i_amar = ms[-1].start() if ms else len(t)
    md = _RE_DEMIKIAN.search(t, i_amar)
    i_pen = md.start() if md else len(t)
    return {"kepala": t[:i_pert], "pertimbangan": t[i_pert:i_amar], "amar": t[i_amar:i_pen], "penutup": t[i_pen:]}


BIDANG = ["kepala", "pertimbangan", "amar", "penutup"]
baris = []
for path in tqdm(PDF_FILES, desc="Ekstraksi teks PDF"):
    mentah = teks_pdf(path)                 # teks per baris (dibutuhkan regex identitas)
    meta = ekstrak_dari_teks(mentah)        # metadata dengan regex (SUBCPMK-3)
    bersih = bersihkan(mentah)              # teks penuh satu paragraf
    seg = segmentasi(bersih)
    nama = os.path.splitext(os.path.basename(path))[0]
    with open(os.path.join(TXT_DIR, nama + ".txt"), "w", encoding="utf-8") as f:
        f.write(bersih)
    baris.append({
        "File": os.path.basename(path), "No Putusan": meta["No Putusan"], "Tahun": meta["Tahun"],
        "Terdakwa": meta["Terdakwa"], "Jenis Kelamin": meta["Jenis Kelamin"], "Pekerjaan": meta["Pekerjaan"],
        "Pidana Penjara": meta["Pidana Penjara"], "Denda": meta["Denda"], "Tanggal Putusan": meta["Tanggal Putusan"],
        "Barang Bukti": meta["Barang Bukti"], "Amar Putusan": meta["Amar Putusan"],
        "teks": bersih, **seg,
    })

df_dok = pd.DataFrame(baris)
df_dok["_n"] = pd.to_numeric(df_dok["No Putusan"].str.extract(r"^(\d+)")[0], errors="coerce")
df_dok["_t"] = pd.to_numeric(df_dok["Tahun"], errors="coerce")
df_dok = df_dok.sort_values(["_t", "_n"]).drop(columns=["_n", "_t"]).reset_index(drop=True)
df_dok.index.name = "doc_id"
N_DOK = len(df_dok)

print(f"\nJumlah dokumen   : {N_DOK}")
print(f"Panjang teks     : rata-rata {df_dok['teks'].str.len().mean():,.0f} karakter, "
      f"total {df_dok['teks'].str.len().sum():,} karakter")
print("Rata-rata panjang bidang (karakter):", {b: int(df_dok[b].str.len().mean()) for b in BIDANG})
cek = sum(len(df_dok.loc[i, "teks"]) == sum(len(df_dok.loc[i, b]) for b in BIDANG) for i in df_dok.index)
print(f"Bidang menutup seluruh teks pada {cek}/{N_DOK} dokumen (tidak ada bagian putusan yang terbuang)")
noise = re.compile(r"Disclaimer|Kepaniteraan Mahkamah|mahkamahagung\.go\.id|Direktori Putusan", re.I)
print("Dokumen dengan sisa watermark/disclaimer:", int(df_dok["teks"].str.contains(noise).sum()))
if df_overview is not None:
    cocok = set(df_overview["No Putusan"].astype(str).str.strip()) == set(df_dok["No Putusan"])
    print("Nomor putusan identik dengan Overview.xlsx:", cocok)
display(df_dok[["No Putusan", "Terdakwa", "Jenis Kelamin", "Pekerjaan", "Pidana Penjara", "Denda"]].head(5))

## 3. Preprocessing bertahap
| Tahap | Tujuan |
| --- | --- |
| Casefolding | huruf besar/kecil tidak membedakan kata |
| Normalisasi angka | `Rp1.000.000.000,00` → `rp 1000000000`; huruf dan angka yang menempel dipisah |
| Normalisasi varian | `shabu`→`sabu`, `narkoba`→`narkotika`, `handphone`→`hp`, `als`→`alias` |
| Tokenisasi | memecah teks menjadi token alfanumerik |
| Stopword removal | daftar Sastrawi + kata rumus putusan (`menimbang`, `mengingat`, `bahwa`, …) |
| Stemming | Sastrawi (algoritma Nazief–Adriani); hasil di-*cache* per kata |

Empat konfigurasi (A–D) dibangun secara kumulatif agar pengaruh tiap tahap dapat dibandingkan pada bagian evaluasi.

In [ ]:
# ===== PREPROCESSING BERTAHAP: casefolding -> normalisasi -> tokenisasi -> stopword -> stemming =====
_stemmer = StemmerFactory().create_stemmer()
STOP_SASTRAWI = set(StopWordRemoverFactory().get_stop_words())
# kata baku/rumus putusan yang hampir selalu ada dan tidak membedakan satu putusan dari yang lain
STOP_DOMAIN = {"menimbang", "mengingat", "bahwa", "tersebut", "sebagaimana", "dimaksud", "halaman",
               "putusan", "direktori", "mahkamah", "agung", "republik", "indonesia"}
STOPWORDS = STOP_SASTRAWI | STOP_DOMAIN

# varian ejaan/istilah yang sama maknanya pada putusan narkotika
VARIAN = {"shabu": "sabu", "narkoba": "narkotika", "handphone": "hp", "ponsel": "hp", "smartphone": "hp",
          "als": "alias"}

_cache_stem = {}


def stem_kata(w):
    if not w.isalpha():
        return w
    s = _cache_stem.get(w)
    if s is None:
        s = _cache_stem[w] = _stemmer.stem(w)
    return s


def normalisasi_teks(t):
    """Casefolding + perapihan angka: 'Rp1.000.000.000,00' -> 'rp 1000000000 00'."""
    t = t.lower()
    t = re.sub(r"(?<=\d)[.,](?=\d{3}(?!\d))", "", t)          # pemisah ribuan
    t = re.sub(r"(?<=[a-z])(?=\d)|(?<=\d)(?=[a-z])", " ", t)  # rp800 -> rp 800 ; 2024pn -> 2024 pn
    return re.sub(r"[^a-z0-9]+", " ", t)


class Praproses:
    """Rangkaian praproses yang dapat dihidupkan/dimatikan per tahap, agar pengaruh tiap tahap bisa diukur."""

    def __init__(self, nama, norm, stop, stem):
        self.nama, self.norm, self.stop, self.stem = nama, norm, stop, stem

    def __call__(self, teks):
        if self.norm:
            toks = normalisasi_teks(teks).split()
            toks = [VARIAN.get(t, t) for t in toks]
            toks = [t for t in toks if not (t.isalpha() and len(t) == 1) and not (t.isdigit() and set(t) == {"0"})]
        else:
            toks = re.findall(r"[a-z0-9]+", teks.lower())
        if self.stop:
            toks = [t for t in toks if t not in STOPWORDS]
        if self.stem:
            toks = [stem_kata(t) for t in toks]
        return toks


KONFIG = {
    "A_dasar":        Praproses("A. Dasar (casefolding + tokenisasi)", norm=False, stop=False, stem=False),
    "B_stopword":     Praproses("B. + Stopword removal",              norm=False, stop=True,  stem=False),
    "C_normalisasi":  Praproses("C. + Normalisasi angka & varian",    norm=True,  stop=True,  stem=False),
    "D_stemming":     Praproses("D. + Stemming (lengkap)",            norm=True,  stop=True,  stem=True),
}

# --- Contoh: satu kalimat amar melewati tiap tahap ---
contoh = df_dok.loc[0, "amar"][:230]
print("TEKS ASLI     :", contoh)
for kode, pp in KONFIG.items():
    print(f"\n{pp.nama}\n  -> {pp(contoh)[:22]}")

# --- Token tiap bidang untuk tiap konfigurasi (seluruh bagian putusan ikut diproses) ---
t0 = time.time()
TOKEN = {kode: {b: [pp(df_dok.loc[i, b]) for i in df_dok.index] for b in BIDANG} for kode, pp in KONFIG.items()}
TOKEN_PENUH = {kode: [sum((TOKEN[kode][b][i] for b in BIDANG), []) for i in range(N_DOK)] for kode in KONFIG}
print(f"\nPraproses seluruh korpus selesai dalam {time.time()-t0:.1f} detik")

# --- Dampak tiap tahap terhadap ukuran korpus ---
stat = []
for kode, pp in KONFIG.items():
    semua = [t for toks in TOKEN_PENUH[kode] for t in toks]
    stat.append({"Tahap": pp.nama, "Total token": len(semua), "Kosakata unik": len(set(semua)),
                 "Token/dokumen": round(len(semua) / N_DOK)})
df_stat = pd.DataFrame(stat)
df_stat["Kosakata vs A (%)"] = (100 * df_stat["Kosakata unik"] / df_stat.loc[0, "Kosakata unik"]).round(1)
display(df_stat)

## 4. Indexing dan pembobotan
**Inverted index** memetakan tiap *term* ke daftar dokumen yang memuatnya (*posting list*), lengkap dengan frekuensi dan posisi kata. Tiga skema pembobotan diimplementasikan:

- **TF mentah**: skor = jumlah kemunculan kata kueri dalam dokumen.
- **TF-IDF + cosine** (model ruang vektor): bobot $w_{t,d} = (1+\ln tf_{t,d}) \times \left(\ln\frac{1+N}{1+df_t}+1\right)$, vektor dinormalisasi L2.
- **BM25** (model probabilistik): $\sum_t \ln\!\left(1+\frac{N-df_t+0.5}{df_t+0.5}\right)\cdot\frac{tf\,(k_1+1)}{tf+k_1(1-b+b\,\frac{dl}{avgdl})}$ dengan $k_1=1.5$, $b=0.75$.
- **BM25 berbobot bidang**: jumlah skor BM25 tiap bidang (kepala, pertimbangan, amar, penutup) dikali bobot bidang.

In [ ]:
# ===== INDEKS TERBALIK (INVERTED INDEX) + PEMBOBOTAN, ditulis sendiri =====
class Indeks:
    """Inverted index: term -> {dokumen: frekuensi (+ posisi)}. Disimpan sebagai matriks sparse dokumen x term."""

    def __init__(self, token_docs, simpan_posisi=False):
        self.N = len(token_docs)
        self.doc_len = np.array([len(t) for t in token_docs], dtype=np.float64)
        self.avgdl = float(self.doc_len.mean()) if self.N else 0.0
        self.vocab, baris, kolom, nilai = {}, [], [], []
        self.posisi = defaultdict(dict) if simpan_posisi else None
        for d, toks in enumerate(token_docs):
            for term, c in Counter(toks).items():
                j = self.vocab.setdefault(term, len(self.vocab))
                baris.append(d); kolom.append(j); nilai.append(c)
            if simpan_posisi:
                tmp = defaultdict(list)
                for p, term in enumerate(toks):
                    tmp[term].append(p)
                for term, ps in tmp.items():
                    self.posisi[term][d] = np.array(ps, dtype=np.int32)
        self.terms = list(self.vocab.keys())
        self.tf = sp.csr_matrix((nilai, (baris, kolom)), shape=(self.N, len(self.vocab)), dtype=np.float64)
        self._csc = self.tf.tocsc()
        self.df = np.asarray((self.tf > 0).sum(axis=0)).ravel()      # document frequency
        self.cf = np.asarray(self.tf.sum(axis=0)).ravel()            # collection frequency
        self._W = None

    # ---- akses posting list ----
    def posting(self, term):
        j = self.vocab.get(term)
        if j is None:
            return np.array([], dtype=int), np.array([])
        a, b = self._csc.indptr[j], self._csc.indptr[j + 1]
        return self._csc.indices[a:b], self._csc.data[a:b]

    def docs_term(self, term):
        return set(self.posting(term)[0].tolist())

    def docs_frasa(self, tokens):
        """Dokumen yang memuat urutan token berurutan (membutuhkan indeks posisi)."""
        if self.posisi is None:
            raise ValueError("Indeks dibuat tanpa posisi.")
        if not tokens:
            return set()
        calon = set.intersection(*[set(self.posisi.get(t, {}).keys()) for t in tokens])
        hasil = set()
        for d in calon:
            awal = self.posisi[tokens[0]][d]
            for p in awal:
                if all((p + i) in set(self.posisi[t][d].tolist()) for i, t in enumerate(tokens[1:], 1)):
                    hasil.add(d)
                    break
        return hasil

    # ---- pembobotan 1: TF mentah ----
    def skor_tf(self, q):
        skor = np.zeros(self.N)
        for term in set(q):
            di, tf = self.posting(term)
            skor[di] += tf
        return skor

    # ---- pembobotan 2: TF-IDF (tf sublinear x idf, ternormalisasi L2) + cosine similarity ----
    def _matriks_tfidf(self):
        if self._W is None:
            self.idf = np.log((1 + self.N) / (1 + self.df)) + 1.0
            w = self.tf.copy()
            w.data = 1.0 + np.log(w.data)
            w = w @ sp.diags(self.idf)
            norma = np.sqrt(np.asarray(w.multiply(w).sum(axis=1)).ravel())
            norma[norma == 0] = 1.0
            self._W = (sp.diags(1.0 / norma) @ w).tocsc()
        return self._W

    def skor_tfidf(self, q):
        W = self._matriks_tfidf()
        cq = Counter(t for t in q if t in self.vocab)
        if not cq:
            return np.zeros(self.N)
        idx = np.array([self.vocab[t] for t in cq])
        wq = np.array([(1.0 + math.log(c)) * self.idf[self.vocab[t]] for t, c in cq.items()])
        wq = wq / np.linalg.norm(wq)
        return np.asarray(W[:, idx] @ wq).ravel()

    # ---- pembobotan 3: BM25 (model probabilistik) ----
    def skor_bm25(self, q, k1=BM25_K1, b=BM25_B):
        skor = np.zeros(self.N)
        for term in set(q):
            j = self.vocab.get(term)
            if j is None:
                continue
            idf = math.log(1.0 + (self.N - self.df[j] + 0.5) / (self.df[j] + 0.5))
            di, tf = self.posting(term)
            panjang = k1 * (1.0 - b + b * self.doc_len[di] / self.avgdl)
            skor[di] += idf * tf * (k1 + 1.0) / (tf + panjang)
        return skor


class MesinCari:
    """Satu konfigurasi praproses + indeks seluruh teks + indeks per bidang."""

    def __init__(self, kode, simpan_posisi=False):
        self.kode, self.pp = kode, KONFIG[kode]
        self.penuh = Indeks(TOKEN_PENUH[kode], simpan_posisi=simpan_posisi)
        self.bidang = {b: Indeks(TOKEN[kode][b]) for b in BIDANG}

    def skor(self, query, metode):
        q = self.pp(query)
        if metode == "tf":
            return self.penuh.skor_tf(q)
        if metode == "tfidf":
            return self.penuh.skor_tfidf(q)
        if metode == "bm25":
            return self.penuh.skor_bm25(q)
        if metode == "bm25_bidang":     # jumlah skor BM25 tiap bidang x bobot bidang
            return sum(BOBOT_BIDANG[b] * self.bidang[b].skor_bm25(q) for b in BIDANG)
        raise ValueError(metode)

    def urutan(self, query, metode):
        s = self.skor(query, metode)
        o = np.lexsort((np.arange(self.penuh.N), -s))
        return [int(d) for d in o if s[d] > 1e-12], s


METODE = {"tf": "TF mentah", "tfidf": "TF-IDF + cosine", "bm25": "BM25", "bm25_bidang": "BM25 berbobot bidang"}

t0 = time.time()
MESIN = {kode: MesinCari(kode) for kode in KONFIG}
print(f"4 indeks (satu per konfigurasi praproses) dibangun dalam {time.time()-t0:.1f} detik")
for kode, m in MESIN.items():
    print(f"  {KONFIG[kode].nama:42} | term: {len(m.penuh.vocab):6,} | posting: {m.penuh.tf.nnz:8,} | avgdl: {m.penuh.avgdl:,.0f}")

# --- Validasi: TF-IDF buatan sendiri harus sama dengan scikit-learn ---
from sklearn.feature_extraction.text import TfidfVectorizer
m = MESIN["D_stemming"]
vek = TfidfVectorizer(analyzer=lambda x: x, smooth_idf=True, sublinear_tf=True, norm="l2")
X = vek.fit_transform(TOKEN_PENUH["D_stemming"])
selisih = 0.0
for qtext in ["terdakwa perempuan", "permufakatan jahat narkotika", "pidana penjara 15 tahun", "barang bukti timbangan digital"]:
    q = m.pp(qtext)
    sk = (X @ vek.transform([q]).T).toarray().ravel()
    selisih = max(selisih, float(np.abs(sk - m.penuh.skor_tfidf(q)).max()))
print(f"\nValidasi TF-IDF vs scikit-learn: selisih skor maksimum = {selisih:.2e}", "(sesuai)" if selisih < 1e-9 else "(TIDAK sesuai)")
assert selisih < 1e-9

## 5. Evaluasi: tahap praproses dan pembobotan mana yang meningkatkan kualitas pencarian?
Tidak tersedia penilaian relevansi manual, sehingga dipakai *silver standard*: untuk tiap kueri bahasa alami, putusan relevan ditentukan otomatis dari atribut hasil ekstraksi (misalnya pekerjaan terdakwa, lama pidana, isi barang bukti). Metrik: **MAP**, **nDCG@10**, **R-Precision**, dan **MRR**.

In [ ]:
# ===== EVALUASI KUALITAS PENCARIAN (judgement relevansi "silver standard") =====
def _kandung(kolom, pola):
    cocok = df_dok[kolom].str.contains(pola, case=False, regex=True, na=False)
    return set(int(i) for i in df_dok.index[cocok])


# Setiap kueri ditulis dalam bahasa alami; putusan yang RELEVAN ditentukan dari atribut hasil ekstraksi
QUERIES = [
    ("terdakwa perempuan",             _kandung("Jenis Kelamin", r"^perempuan"),         "jenis kelamin terdakwa"),
    ("terdakwa wiraswasta",            _kandung("Pekerjaan", r"wiraswasta"),              "pekerjaan terdakwa"),
    ("terdakwa petani pekebun",        _kandung("Pekerjaan", r"petani"),                  "pekerjaan terdakwa"),
    ("terdakwa mengurus rumah tangga", _kandung("Pekerjaan", r"rumah tangga"),            "pekerjaan terdakwa"),
    ("terdakwa karyawan swasta",       _kandung("Pekerjaan", r"karyawan"),                "pekerjaan terdakwa"),
    ("terdakwa pelajar mahasiswa",     _kandung("Pekerjaan", r"pelajar"),                 "pekerjaan terdakwa"),
    ("pidana penjara 15 tahun",        _kandung("Pidana Penjara", r"^15 "),               "lama pidana penjara"),
    ("denda 800 juta rupiah",          _kandung("Denda", r"800\.000\.000"),               "besar denda"),
    ("permufakatan jahat narkotika",   _kandung("Amar Putusan", r"permufakatan jahat"),   "amar: permufakatan jahat"),
    ("perantara jual beli narkotika",  _kandung("Amar Putusan", r"perantara"),            "amar: perantara"),
    ("barang bukti sepeda motor",      _kandung("Barang Bukti", r"motor"),                "barang bukti"),
    ("barang bukti timbangan digital", _kandung("Barang Bukti", r"timbangan"),            "barang bukti"),
]
QUERIES = [q for q in QUERIES if len(q[1]) > 0]
display(pd.DataFrame([{"Kueri": q, "Jumlah putusan relevan": len(r), "Dasar penilaian": d} for q, r, d in QUERIES]))


def metrik(urutan, relevan):
    rel = set(relevan)
    benar, ap = 0, 0.0
    for i, d in enumerate(urutan, 1):
        if d in rel:
            benar += 1
            ap += benar / i
    ap = ap / len(rel)
    R = len(rel)
    r_prec = sum(d in rel for d in urutan[:R]) / R
    dcg = sum(1.0 / math.log2(i + 1) for i, d in enumerate(urutan[:10], 1) if d in rel)
    idcg = sum(1.0 / math.log2(i + 1) for i in range(1, min(R, 10) + 1))
    rr = next((1.0 / i for i, d in enumerate(urutan, 1) if d in rel), 0.0)
    return {"AP": ap, "R-Precision": r_prec, "nDCG@10": dcg / idcg, "RR": rr}


def evaluasi(mesin, metode):
    return pd.DataFrame([metrik(mesin.urutan(q, metode)[0], r) for q, r, _ in QUERIES]).mean()


baris_eval = []
for kode, mesin in MESIN.items():
    for metode, nama_metode in METODE.items():
        h = evaluasi(mesin, metode)
        baris_eval.append({"Praproses": KONFIG[kode].nama, "Pembobotan": nama_metode, "kode": kode, "metode": metode,
                           "MAP": h["AP"], "nDCG@10": h["nDCG@10"], "R-Precision": h["R-Precision"], "MRR": h["RR"]})
df_eval = pd.DataFrame(baris_eval).sort_values(["MAP", "nDCG@10"], ascending=False).reset_index(drop=True)
display(df_eval.drop(columns=["kode", "metode"]).round(3))

print("\nMAP menurut tahap praproses (baris) dan pembobotan (kolom):")
display(df_eval.pivot(index="Praproses", columns="Pembobotan", values="MAP").round(3))

# --- grafik perbandingan ---
piv = df_eval.pivot(index="Praproses", columns="Pembobotan", values="MAP")[[METODE[m] for m in METODE]]
ax = piv.plot(kind="bar", figsize=(11, 4.8), width=0.8)
ax.set_ylabel("MAP (12 kueri)"); ax.set_xlabel("")
ax.set_title("Pengaruh praproses dan pembobotan terhadap kualitas pencarian")
plt.xticks(rotation=15, ha="right"); plt.legend(title="Pembobotan", fontsize=8); plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "perbandingan_metode.png"), dpi=150)
plt.show()

# --- konfigurasi terbaik ---
BEST = df_eval.iloc[0]
KODE_TERBAIK, METODE_TERBAIK = BEST["kode"], BEST["metode"]
print(f"\nKonfigurasi terbaik: {BEST['Praproses']}  +  {BEST['Pembobotan']}  (MAP = {BEST['MAP']:.3f}, nDCG@10 = {BEST['nDCG@10']:.3f})")
dasar = df_eval[(df_eval.kode == "A_dasar") & (df_eval.metode == "tf")].iloc[0]
print(f"Pembanding dasar   : {dasar['Praproses']}  +  {dasar['Pembobotan']}  (MAP = {dasar['MAP']:.3f}, nDCG@10 = {dasar['nDCG@10']:.3f})")

# --- rincian per kueri pada konfigurasi terbaik ---
rinci = []
for q, rel, dasar_nilai in QUERIES:
    urut, _ = MESIN[KODE_TERBAIK].urutan(q, METODE_TERBAIK)
    m_ = metrik(urut, rel)
    pertama = next((i for i, d in enumerate(urut, 1) if d in rel), None)
    rinci.append({"Kueri": q, "Relevan": len(rel), "AP": round(m_["AP"], 3), "R-Precision": round(m_["R-Precision"], 3),
                  "Peringkat relevan pertama": pertama})
display(pd.DataFrame(rinci))
print("Catatan: relevansi ditentukan otomatis dari atribut hasil ekstraksi (silver standard), bukan penilaian manual; "
      "korpus kecil (50 putusan) dan konfigurasi dipilih pada kueri yang sama, sehingga angka bersifat indikatif.")

## 6. Statistik indeks final

In [ ]:
# ===== INDEKS FINAL (dengan posisi kata) DAN STATISTIKNYA =====
MESIN_FINAL = MesinCari(KODE_TERBAIK, simpan_posisi=True)
IDX = MESIN_FINAL.penuh
print(f"Indeks final : {KONFIG[KODE_TERBAIK].nama} | metode pencarian bawaan: {METODE[METODE_TERBAIK]}")
print(f"Dokumen (N)  : {IDX.N} | kosakata: {len(IDX.vocab):,} | total token: {int(IDX.doc_len.sum()):,} | "
      f"rata-rata panjang dokumen: {IDX.avgdl:,.0f} token | jumlah posting: {IDX.tf.nnz:,}")

# --- contoh posting list (inverted index) ---
for term in ([MESIN_FINAL.pp("sabu")[0], MESIN_FINAL.pp("perempuan")[0]]):
    di, tf = IDX.posting(term)
    ps = {int(d): [int(x) for x in IDX.posisi[term][int(d)][:4]] for d in di[:3]}
    print(f"\nTerm '{term}': df = {len(di)} dari {IDX.N} dokumen | cf = {int(tf.sum())}")
    print("  posting (doc_id: posisi awal):", ps)

# --- 15 term terpenting menurut df dan menurut idf ---
df_term = pd.DataFrame({"term": IDX.terms, "df": IDX.df, "cf": IDX.cf.astype(int)})
df_term["idf"] = np.log(1.0 + (IDX.N - df_term["df"] + 0.5) / (df_term["df"] + 0.5))
print("\nTerm dengan df tertinggi (muncul di hampir semua putusan -> daya beda rendah):")
display(df_term.sort_values(["df", "cf"], ascending=False).head(12).reset_index(drop=True))
print("Term dengan frekuensi koleksi tertinggi:")
display(df_term.sort_values("cf", ascending=False).head(12).reset_index(drop=True))

# --- hukum Zipf: peringkat vs frekuensi ---
frek = np.sort(IDX.cf)[::-1]
plt.figure(figsize=(6, 4))
plt.loglog(np.arange(1, len(frek) + 1), frek)
plt.xlabel("peringkat term (log)"); plt.ylabel("frekuensi koleksi (log)")
plt.title("Distribusi frekuensi term (hukum Zipf)")
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, "zipf.png"), dpi=150); plt.show()

## 7. Mesin pencari
Indeks final memakai konfigurasi terbaik dari evaluasi. Tersedia tiga cara mencari: **peringkat** (`cari`), **Boolean** (`AND`, `OR`, `NOT`), dan **frasa** (dalam tanda kutip). Cuplikan teks menandai kata kueri dengan « ».

In [ ]:
# ===== MESIN PENCARI: ranked retrieval (TF / TF-IDF / BM25) dan Boolean + frasa =====
def cuplikan(teks, query, lebar=110):
    """Potongan teks di sekitar kata kueri; kata yang cocok ditandai dengan « »."""
    kata = [w for w in re.findall(r"[a-z0-9]+", query.lower()) if w not in STOPWORDS and len(w) > 1]
    for w in sorted(set(kata), key=len, reverse=True):
        m = re.search(r"\b" + re.escape(w) + r"\w*", teks, re.I)
        if m:
            a, b = max(0, m.start() - lebar), min(len(teks), m.end() + lebar)
            return ("… " if a > 0 else "") + teks[a:m.start()] + "«" + teks[m.start():m.end()] + "»" + teks[m.end():b] + (" …" if b < len(teks) else "")
    return teks[:2 * lebar] + " …"


def cari(query, metode=None, k=TOP_K, mesin=None):
    """Pencarian peringkat. Mengembalikan DataFrame k putusan teratas beserta cuplikan teks."""
    mesin = mesin or MESIN_FINAL
    metode = metode or METODE_TERBAIK
    urut, skor = mesin.urutan(query, metode)
    if not urut:
        return pd.DataFrame(columns=["Peringkat", "Skor", "No Putusan", "Terdakwa", "Pidana Penjara", "Cuplikan"])
    baris = []
    for r, d in enumerate(urut[:k], 1):
        baris.append({"Peringkat": r, "Skor": round(float(skor[d]), 3), "No Putusan": df_dok.loc[d, "No Putusan"],
                      "Terdakwa": df_dok.loc[d, "Terdakwa"][:38], "Pidana Penjara": df_dok.loc[d, "Pidana Penjara"],
                      "Cuplikan": cuplikan(df_dok.loc[d, "teks"], query)})
    return pd.DataFrame(baris)


def _operand_docs(operand, mesin):
    idx = mesin.penuh
    if operand.startswith('"'):                           # frasa -> cocok urutan kata (posisi)
        toks = mesin.pp(operand.strip('"'))
        return idx.docs_frasa(toks) if toks else None
    toks = mesin.pp(operand)
    return set.intersection(*[idx.docs_term(t) for t in toks]) if toks else None


def cari_boolean(query, mesin=None):
    """Model Boolean: operator AND, OR, NOT (huruf kapital) dan frasa dalam tanda kutip.
    Contoh: 'perempuan AND "rumah tangga"', '"permufakatan jahat"', '"sepeda motor" NOT perempuan'.
    Tanpa operator, kata-kata digabung dengan AND. AND diproses lebih dulu daripada OR."""
    mesin = mesin or MESIN_FINAL
    grup, neg = [[]], False
    for it in re.findall(r'"[^"]+"|\S+', query):
        if it == "OR":
            grup.append([]); neg = False
        elif it == "AND":
            continue
        elif it == "NOT":
            neg = True
        else:
            grup[-1].append((neg, it)); neg = False
    hasil = set()
    for g in grup:
        pos, ng = [], []
        for negatif, op in g:
            d = _operand_docs(op, mesin)
            if d is not None:
                (ng if negatif else pos).append(d)
        if not pos and not ng:
            continue
        cur = set.intersection(*pos) if pos else set(range(mesin.penuh.N))
        for d in ng:
            cur -= d
        hasil |= cur
    return sorted(hasil)


def tampil_boolean(query):
    ids = cari_boolean(query)
    print(f"\nKueri Boolean: {query}   ->   {len(ids)} putusan")
    if ids:
        display(df_dok.loc[ids[:8], ["No Putusan", "Terdakwa", "Pekerjaan", "Pidana Penjara"]])


# --- Demonstrasi pencarian peringkat (metode terbaik hasil evaluasi) ---
for q in ["terdakwa perempuan mengurus rumah tangga", "permufakatan jahat narkotika", "pidana penjara 15 tahun",
          "sepeda motor"]:
    print(f"\n>>> Kueri: '{q}'  [{METODE[METODE_TERBAIK]}]")
    display(cari(q, k=3))

# --- Perbandingan metode pada satu kueri yang sama ---
q = "terdakwa pelajar mahasiswa"
print(f"\n>>> Perbandingan metode untuk kueri '{q}' (3 teratas tiap metode):")
for metode, nama in METODE.items():
    urut, skor = MESIN_FINAL.urutan(q, metode)
    print(f"  {nama:22}:", [(df_dok.loc[d, 'No Putusan'].split('/PN')[0], round(float(skor[d]), 2)) for d in urut[:3]])

# --- Demonstrasi Boolean dan frasa ---
tampil_boolean('"permufakatan jahat"')                      # frasa (urutan kata via indeks posisi)
tampil_boolean('perempuan AND "rumah tangga"')              # AND
tampil_boolean('"sepeda motor" NOT perempuan')              # NOT
tampil_boolean('"permufakatan jahat" OR pelajar')           # OR

## 8. Menyimpan hasil
Indeks, kamus term, korpus hasil praproses, ringkasan putusan, dan hasil evaluasi disimpan ke folder keluaran dan dikemas dalam satu berkas ZIP.

In [ ]:
# ===== MENYIMPAN HASIL DAN RINGKASAN =====
# 1) indeks (data murni, dapat dibuka tanpa kode notebook)
with open(os.path.join(OUT_DIR, "indeks_putusan.pkl"), "wb") as f:
    pickle.dump({"konfigurasi": KONFIG[KODE_TERBAIK].nama, "metode_bawaan": METODE_TERBAIK, "terms": IDX.terms,
                 "tf_dokumen_x_term": IDX.tf, "df": IDX.df, "doc_len": IDX.doc_len,
                 "no_putusan": df_dok["No Putusan"].tolist()}, f)
# 2) kamus term, korpus hasil praproses, ringkasan dokumen, hasil evaluasi
df_term.sort_values("df", ascending=False).to_csv(os.path.join(OUT_DIR, "kamus_term.csv"), index=False)
pd.DataFrame({"doc_id": df_dok.index, "No Putusan": df_dok["No Putusan"],
              "token_hasil_praproses": [" ".join(t) for t in TOKEN_PENUH[KODE_TERBAIK]]}
             ).to_csv(os.path.join(OUT_DIR, "korpus_praproses.csv"), index=False, encoding="utf-8-sig")
df_dok.drop(columns=["teks"] + BIDANG).to_csv(os.path.join(OUT_DIR, "ringkasan_putusan.csv"), index=False, encoding="utf-8-sig")
df_eval.drop(columns=["kode", "metode"]).round(4).to_csv(os.path.join(OUT_DIR, "hasil_evaluasi.csv"), index=False)

# 3) kemas semuanya (kecuali salinan PDF dataset yang sudah ada di GitHub)
paket = os.path.join(os.path.dirname(OUT_DIR), "hasil_subcpmk4")
sementara = os.path.join(os.path.dirname(OUT_DIR), "_paket_subcpmk4")
shutil.rmtree(sementara, ignore_errors=True)
shutil.copytree(OUT_DIR, sementara, ignore=shutil.ignore_patterns("dataset"))
zip_path = shutil.make_archive(paket, "zip", sementara)
shutil.rmtree(sementara, ignore_errors=True)
print("Berkas keluaran :", sorted(f for f in os.listdir(OUT_DIR) if f not in ("dataset", "txt")), "+ folder txt/ (50 TXT)")
print(f"Paket ZIP       : {zip_path} ({os.path.getsize(zip_path)/1e6:.1f} MB)")

if SIMPAN_KE_DRIVE and DI_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    tujuan = f"/content/drive/MyDrive/{DRIVE_FOLDER}"
    shutil.copytree(OUT_DIR, tujuan, dirs_exist_ok=True, ignore=shutil.ignore_patterns("dataset"))
    print("Disalin ke Drive:", tujuan)

# 4) ringkasan akhir
print("\n=== RINGKASAN ===")
print(f"Dokumen diindeks            : {N_DOK} putusan (seluruh bagian: kepala, pertimbangan, amar, penutup)")
print(f"Kosakata                    : {len(MESIN['A_dasar'].penuh.vocab):,} term (dasar) -> {len(IDX.vocab):,} term ({KONFIG[KODE_TERBAIK].nama})")
print(f"Konfigurasi terbaik         : {KONFIG[KODE_TERBAIK].nama} + {METODE[METODE_TERBAIK]}")
print(f"MAP konfigurasi terbaik     : {BEST['MAP']:.3f}  (pembanding dasar: {dasar['MAP']:.3f})")

if DI_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print("Unduh otomatis dilewati:", e)